In [33]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool, InjectedToolArg
from langchain_core.messages import HumanMessage
import requests
from typing import Annotated

Tool Create

In [40]:
# first tool will fetch us the conversion factor
# and second tool will give the results after conversion

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """
    This function fetches the currency conversion factor between a given base currency and a target base_currency
    """
    url = f'https://v6.exchangerate-api.com/v6/29b17ae2f01ee12d317269d0/pair/{base_currency}/{target_currency}'
    response = requests.get(url)
    return response.json()

@tool
def convert(base_currency_value: float, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """
    Given a currency conversion rate this function calculates the target currency value from a given base currency value
    """
    return base_currency_value * conversion_rate

In [41]:
get_conversion_factor.invoke({'base_currency':'USD', 'target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1786924801,
 'time_last_update_utc': 'Mon, 17 Aug 2026 00:00:01 +0000',
 'time_next_update_unix': 1787011201,
 'time_next_update_utc': 'Tue, 18 Aug 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.6696}

In [42]:
convert.invoke({'base_currency_value':10, 'conversion_rate':95.6696})

956.696

In [43]:
# tool binding
llm = ChatGoogleGenerativeAI(model = 'gemini-3.1-flash-lite')
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [52]:
messages = [HumanMessage('Convert 14 USD to INR after getting the current conversion rate.')]

In [53]:
messages

[HumanMessage(content='Convert 14 USD to INR after getting the current conversion rate.', additional_kwargs={}, response_metadata={})]

In [54]:
ai_message = llm_with_tools.invoke(messages)

In [59]:
tool_call = ai_message.tool_calls
print(tool_call)
print(type(tool_call))

[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_1789782', 'type': 'tool_call'}]
<class 'list'>


In [65]:
# execute the tool
tool_message = get_conversion_factor.invoke(tool_call[0]["args"])
print(tool_message)


{'result': 'success', 'documentation': 'https://www.exchangerate-api.com/docs', 'terms_of_use': 'https://www.exchangerate-api.com/terms', 'time_last_update_unix': 1786924801, 'time_last_update_utc': 'Mon, 17 Aug 2026 00:00:01 +0000', 'time_next_update_unix': 1787011201, 'time_next_update_utc': 'Tue, 18 Aug 2026 00:00:01 +0000', 'base_code': 'USD', 'target_code': 'INR', 'conversion_rate': 95.6696}
